# InceptionV1 feature visualization — Kaggle GPU gallery

Activation maximization on InceptionV1 (the canonical mechanistic-interpretability
model): synthesise, from noise, the image that most excites each channel — "what does
this unit want to see?". Recipe is the lucid one (Fourier param + colour decorrelation +
transform robustness), which is what makes the images look like features, not static.

**Self-contained — no data upload needed.** GoogLeNet weights download from torchvision.
- Settings → **Accelerator: GPU T4**
- Settings → **Internet: On** (needed for the weight download; requires a phone-verified
  Kaggle account)
- Then **Run All**.

**Outputs** land in `/kaggle/working/featureviz/` (one PNG per channel + per-layer and
master montages + `activations.csv`) and are zipped to `/kaggle/working/featureviz.zip`
for a single download. How to download is in the last cell.

In [ ]:
import os, math, time, shutil, csv
import numpy as np
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from torchvision import models
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = models.googlenet(weights=models.GoogLeNet_Weights.IMAGENET1K_V1).to(device).eval()
for p in model.parameters():
    p.requires_grad_(False)
modules = dict(model.named_modules())
print('device:', device)

In [ ]:
# --- the lucid feature-visualization recipe ---
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
COLOR_CORR = torch.tensor([[0.26, 0.09, 0.02], [0.27, 0.0, -0.05], [0.27, -0.09, 0.03]], device=device)
COLOR_CORR = COLOR_CORR / torch.linalg.norm(COLOR_CORR, dim=1).max()

def rfft_freqs(h, w):
    fy = np.fft.fftfreq(h)[:, None]; fx = np.fft.rfftfreq(w)[None, :]
    return np.sqrt(fy ** 2 + fx ** 2)

class FourierImage:
    def __init__(self, size, sd=0.01):
        freqs = rfft_freqs(size, size)
        scale = (1.0 / np.maximum(freqs, 1.0 / size)) * math.sqrt(size * size)
        self.scale = torch.tensor(scale, dtype=torch.float32, device=device)[None, :, :]
        self.spectrum = (torch.randn(3, *freqs.shape, 2, device=device) * sd).requires_grad_(True)
        self.size = size
    def parameters(self):
        return [self.spectrum]
    def image(self):
        s = self.spectrum * self.scale[..., None]
        c = torch.complex(s[..., 0], s[..., 1])
        img = torch.fft.irfft2(c, s=(self.size, self.size)) / 4.0
        t = torch.einsum('chw,dc->dhw', img, COLOR_CORR)
        return torch.sigmoid(t)[None]

def augment(img, pad=12):
    img = F.pad(img, [pad] * 4, mode='reflect')
    angle = float(torch.empty(1).uniform_(-10, 10)); scale = float(torch.empty(1).uniform_(0.9, 1.1))
    tx = int(torch.randint(-8, 9, (1,))); ty = int(torch.randint(-8, 9, (1,)))
    img = TF.affine(img, angle=angle, translate=[tx, ty], scale=scale, shear=[0.0, 0.0],
                    interpolation=TF.InterpolationMode.BILINEAR)
    return img[:, :, pad:-pad, pad:-pad]

def visualize(layer, channel, size=224, steps=384, lr=0.05):
    mod = modules[layer]; cap = {}
    h = mod.register_forward_hook(lambda m, i, o: cap.__setitem__('a', o))
    param = FourierImage(size); opt = torch.optim.Adam(param.parameters(), lr=lr)
    try:
        for _ in range(steps):
            opt.zero_grad()
            x = (augment(param.image()) - IMAGENET_MEAN) / IMAGENET_STD
            model(x)
            loss = -cap['a'][0, channel].mean(); loss.backward(); opt.step()
        with torch.no_grad():
            img = np.clip(param.image()[0].permute(1, 2, 0).cpu().numpy(), 0, 1)
        return img, float(-loss.detach())
    finally:
        h.remove()

def channel_count(layer):
    cap = {}; h = modules[layer].register_forward_hook(lambda m, i, o: cap.__setitem__('a', o))
    model(torch.zeros(1, 3, 224, 224, device=device)); h.remove()
    return cap['a'].shape[1]
print('recipe ready')

In [ ]:
# --- CONFIG: tune how big the run is, then read the ETA before the full loop ---
LAYERS = ['inception3a', 'inception3b', 'inception4a', 'inception4b', 'inception4c',
          'inception4d', 'inception4e', 'inception5a', 'inception5b']  # shallow -> deep
CHANNELS_PER_LAYER = 12   # evenly spaced across each layer's channel range
SIZE = 224                # raise to 256/320 for prettier (slower) images
STEPS = 384               # optimization steps per channel

n_total = len(LAYERS) * CHANNELS_PER_LAYER
t0 = time.time(); _ = visualize(LAYERS[-1], 0, size=SIZE, steps=STEPS); dt = time.time() - t0
print(f'~{dt:.1f}s per channel on this GPU  ->  est. {n_total * dt / 60:.1f} min for {n_total} images')
print('Adjust CHANNELS_PER_LAYER / SIZE / STEPS above if that is too long, then run the next cell.')

In [ ]:
# --- the big run: one PNG per channel + per-layer montages + master montage ---
OUT = '/kaggle/working/featureviz'
os.makedirs(OUT, exist_ok=True)
records, per_layer_imgs = [], {}
run_start = time.time()
for li, layer in enumerate(LAYERS):
    total = channel_count(layer)
    chans = sorted(set(np.linspace(0, total - 1, CHANNELS_PER_LAYER).astype(int).tolist()))
    imgs = []
    for ch in chans:
        img, act = visualize(layer, ch, size=SIZE, steps=STEPS)
        plt.imsave(f'{OUT}/{layer}_ch{ch:04d}.png', img)
        imgs.append((ch, img, act)); records.append({'layer': layer, 'channel': int(ch), 'activation': round(act, 3)})
    per_layer_imgs[layer] = imgs
    # per-layer montage
    fig, ax = plt.subplots(1, len(imgs), figsize=(1.8 * len(imgs), 2.1))
    ax = np.atleast_1d(ax)
    for a, (ch, img, act) in zip(ax, imgs):
        a.imshow(img); a.set_title(f'ch{ch}\n{act:.1f}', fontsize=8); a.set_xticks([]); a.set_yticks([])
    fig.suptitle(layer, fontsize=11); fig.tight_layout(rect=(0, 0, 1, 0.9))
    fig.savefig(f'{OUT}/montage_{li}_{layer}.png', dpi=140); plt.close(fig)
    print(f'[{li+1}/{len(LAYERS)}] {layer}: {len(imgs)} channels done  ({(time.time()-run_start)/60:.1f} min elapsed)')

with open(f'{OUT}/activations.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['layer', 'channel', 'activation']); w.writeheader(); w.writerows(records)
print('all channels rendered')

In [ ]:
# --- master montage: one row per layer, shallow (top) -> deep (bottom) ---
ncol = max(len(v) for v in per_layer_imgs.values())
fig, axes = plt.subplots(len(LAYERS), ncol, figsize=(1.8 * ncol, 2.0 * len(LAYERS)))
axes = np.atleast_2d(axes)
for r, layer in enumerate(LAYERS):
    imgs = per_layer_imgs[layer]
    for c in range(ncol):
        a = axes[r, c]; a.set_xticks([]); a.set_yticks([])
        if c < len(imgs):
            ch, img, act = imgs[c]; a.imshow(img); a.set_title(f'ch{ch}', fontsize=7)
        else:
            a.axis('off')
    axes[r, 0].set_ylabel(layer, fontsize=10, rotation=0, ha='right', va='center', labelpad=34)
fig.suptitle('InceptionV1 feature visualization — shallow (top) to deep (bottom)', fontsize=13)
fig.tight_layout(rect=(0.05, 0, 1, 0.98))
fig.savefig(f'{OUT}/master_montage.png', dpi=150); plt.show()

In [ ]:
# --- bundle everything into one downloadable zip ---
shutil.make_archive('/kaggle/working/featureviz', 'zip', OUT)
print('Done. Download the results in one of two ways:')
print('  A) Right sidebar -> Output -> hover featureviz.zip -> the download icon.')
print('  B) Notebook menu -> File -> Download (or, after Save Version, the Output tab).')
print('Then put featureviz.zip on your Desktop and tell Claude to import it into the worktree.')